# ConnectPlus — Customer Churn Prediction

**Objective:** identify customers at elevated churn risk and translate predictive signals into retention actions.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

df=pd.read_csv("data/subscription_customers.csv")
df.head()

## 1. Data Audit

In [ ]:
print("Shape:",df.shape)
print("\nMissing values:\n",df.isna().sum())
print("\nTarget distribution:\n",df.churned.value_counts(normalize=True).round(3))

## 2. Exploratory Analysis

In [ ]:
print(df.groupby("contract_type").churned.agg(["mean","count"]).sort_values("mean",ascending=False).round(3))
print("\nSupport-ticket relationship:")
print(df.groupby(pd.cut(df.support_tickets,[-1,0,2,4,100])).churned.mean().round(3))
print("\nSatisfaction relationship:")
print(df.groupby(pd.cut(df.satisfaction_score,[0,4,6,8,10.1])).churned.mean().round(3))

## 3. Reproducible Modeling Pipeline

In [ ]:
X=df.drop(columns="churned"); y=df.churned
cat_features=["contract_type"]
num_features=[c for c in X.columns if c not in cat_features+["customer_id"]]

preprocessor=ColumnTransformer([
    ("numeric",Pipeline([("imputer",SimpleImputer(strategy="median")),("scaler",StandardScaler())]),num_features),
    ("categorical",OneHotEncoder(handle_unknown="ignore"),cat_features)
])
pipeline=Pipeline([
    ("preprocessor",preprocessor),
    ("model",LogisticRegression(max_iter=1000,class_weight="balanced",random_state=42))
])
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.20,stratify=y,random_state=42)
pipeline.fit(X_train,y_train)
pred=pipeline.predict(X_test); proba=pipeline.predict_proba(X_test)[:,1]
print("ROC-AUC:",round(roc_auc_score(y_test,proba),3))
print(classification_report(y_test,pred))

## 4. Confusion Matrix

In [ ]:
cm=confusion_matrix(y_test,pred)
ConfusionMatrixDisplay(cm).plot()
plt.title("Churn Classification — Test Set")
plt.tight_layout()

## 5. Customer Risk Scoring

In [ ]:
scored=df.drop(columns="churned").copy()
scored["churn_probability"]=pipeline.predict_proba(scored)[:,1]
scored["risk_segment"]=pd.cut(
    scored.churn_probability,bins=[-.01,.30,.60,1.01],
    labels=["Lower Risk","Medium Risk","Higher Risk"])
scored[["customer_id","churn_probability","risk_segment"]].sort_values(
    "churn_probability",ascending=False).head(20)

## Retention playbook

**High risk + support friction:** service recovery.

**High risk + low satisfaction:** customer-success outreach and root-cause investigation.

**High risk + monthly contract:** test longer-term plans or value-based retention offers.

**Measurement:** evaluate retention lift and intervention cost with controlled experiments.